# B2-024-gpu-scientific-ml-capstone — Practice p19 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** scientific-ml-inverse-problems

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "simulated_pairs": 0, "hidden": 64,
              "steps": 1500, "batch_size": None, "lr": 3e-3}
RESIDUAL_BOUND = 0.06
ERROR_BOUND = 0.03

## Solution

**Model.** The magnitudes span orders of magnitude and the noise is multiplicative, so the network sees standardized $\log m$ (Session 2's mechanism): $\log m=\log q-\log(r^2+h^2)+\log(1+0.03\varepsilon)$ turns the strength into an additive offset and the noise into roughly constant-scale additive noise. The statistics are registered **buffers** computed from the training magnitudes, so they live inside the model, move with it, and the model receives the stored rows unchanged (which is what makes the seam hash check possible).

**Training.** Exactly 1,500 full-batch Adam steps on the MSE of $(x,y,q)$ under `CPU_CONFIG`, all three targets being of order 1. Device, precision, network width, step count, batch size, learning rate, and the number of simulated pairs all come from the config dict, so the L4 run changes only the dictionary. A forward pre-hook hashes the rows of the first training-mode call.

**Physics residual.** For a predicted source $(\hat x,\hat y,\hat q)$, the p10 forward model gives the readings that source *would* produce; comparing them with the observed readings, $\lVert\mathcal F(\hat s,\hat q)-m\rVert_2/\lVert m\rVert_2$, needs no true source. With true sources this residual is the noise floor (about 3% multiplicative noise).

In [ ]:
from contextlib import nullcontext

CONFIG = CPU_CONFIG                    # the accelerator extension replaces only this dict

x, y = capstone_data.train_rows("inverse")
xv, yv = capstone_data.val_rows("inverse")


class LearnedInverse(nn.Module):
    def __init__(self, train_magnitudes, hidden=64):
        super().__init__()
        logm = torch.log(train_magnitudes)
        self.register_buffer("mu", logm.mean(dim=0))
        self.register_buffer("sd", logm.std(dim=0))
        self.net = nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 3))

    def forward(self, m):
        return self.net((torch.log(m) - self.mu) / self.sd)


def forward_field(source_xy, strength, sensors_xy, height=capstone_data.SENSOR_HEIGHT):
    r2 = ((source_xy.double()[:, None, :] - sensors_xy.double()[None, :, :]) ** 2).sum(dim=-1)
    return strength.double()[:, None] / (r2 + height ** 2)


def relative_residual(pred, magnitudes):
    field = forward_field(pred[:, :2], pred[:, 2], capstone_data.SENSORS_XY)
    m = magnitudes.double()
    return torch.linalg.vector_norm(field - m, dim=1) / torch.linalg.vector_norm(m, dim=1)


SEAM = set()
seam_state = {"armed": True}


def seam_hook(module, args):
    if module.training and seam_state["armed"]:
        seam_state["armed"] = False
        SEAM.update(capstone_data.split_of("inverse", row) for row in args[0].detach().cpu())


def train_inverse(model, x, y, optimizer, config):
    device = torch.device(config["device"])
    x, y = x.to(device), y.to(device)
    amp = config["amp_dtype"]
    model.train()
    losses = []
    for _ in range(config["steps"]):
        if config["batch_size"] is None:
            xb, yb = x, y
        else:
            idx = torch.randint(0, x.shape[0], (config["batch_size"],))
            xb, yb = x[idx], y[idx]
        optimizer.zero_grad(set_to_none=True)
        with nullcontext() if amp is None else torch.autocast(device_type=device.type, dtype=amp):
            loss = F.mse_loss(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


def training_pool(config):
    if config["simulated_pairs"] == 0:
        return x, y
    xs, ys = capstone_data.simulate("inverse", config["simulated_pairs"], seed=3)
    return torch.cat([x, xs]), torch.cat([y, ys])


x_fit, y_fit = training_pool(CONFIG)
device = torch.device(CONFIG["device"])
torch.manual_seed(SEED)
model = LearnedInverse(x_fit, CONFIG["hidden"]).to(device)
model.register_forward_pre_hook(seam_hook)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
losses = train_inverse(model, x_fit, y_fit, optimizer, CONFIG)

model.eval()
with torch.no_grad():
    val_pred = model(xv.to(device)).cpu()
val_error = capstone_data.position_error_per_example(val_pred, yv).mean().item()
val_residual = relative_residual(val_pred, xv).mean().item()
true_residual = relative_residual(yv, xv).mean().item()
tikhonov_val = capstone_data.position_error_per_example(capstone_data.tikhonov_grid_predict(xv), yv).mean().item()
print(f"loss {losses[0]:.4f} -> {losses[-1]:.6f} (ratio {losses[-1] / losses[0]:.5f}) | seam {SEAM}")
print(f"validation position error {val_error:.4f} (grid Tikhonov {tikhonov_val:.4f})")
print(f"mean relative residual: predictions {val_residual:.4f}, true sources (noise floor) {true_residual:.4f}")

**Locked test score (one call, after all fitting).**

In [ ]:
baseline = capstone_data.baseline_score("inverse")


def predict_fn(magnitudes):
    model.eval()
    with torch.no_grad():
        return model(magnitudes.to(device)).cpu()


assert capstone_data.test_call_count("inverse") == 0
test = capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")
assert capstone_data.test_call_count("inverse") == 1
print(f"learned inverse: test position error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]")
print(f"grid-Tikhonov baseline: {baseline.score:.4f} [{baseline.ci_low:.4f}, {baseline.ci_high:.4f}]")

**What the residual can and cannot certify.**
The residual only needs the observed readings and the known forward model: we push the predicted source through $\mathcal F$ and compare with what the sensors measured, so it can be computed for any input, including ones with no known answer.
A residual near the noise floor (here about $0.03$ against a floor of about $0.028$) says the prediction *explains the data* as well as the true source does.
It does not prove the position is right, because different sources can produce nearly the same eight readings (a source slightly farther away but stronger, or an error along a direction the sensors barely resolve) and the 3% noise hides such differences; only the position error against true sources, measured on validation and once on the test set, checks that.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, scale the same `LearnedInverse` / `train_inverse` code by passing `L4_CONFIG` in place of `CPU_CONFIG`. It has exactly the keys of `CPU_CONFIG`, so only the dictionary changes: `simulated_pairs` adds 200,000 training pairs from the known forward process (`capstone_data.simulate("inverse", 200000, seed=3)`), `hidden` widens the network, and `batch_size` switches on mini-batches drawn from the global generator. Keep the seam rule "no `val` or `test` rows" (simulated rows hash to `None`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": None, "simulated_pairs": 200000, "hidden": 256,
             "steps": 20000, "batch_size": 4096, "lr": 1e-3}
```

**What to observe.** Validation position error and mean residual as the number of simulated pairs grows (does the residual approach the noise floor?); whether `bfloat16` autocast changes the error of this small regression network (try it, then decide); and the error near the domain edges.

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
assert len(losses) == 1500
assert losses[-1] <= 0.01 * losses[0]
assert SEAM == {"train"}
assert val_pred.shape == (200, 3)
assert val_residual <= RESIDUAL_BOUND
assert capstone_data.test_call_count("inverse") == 1
assert test.score < baseline.score and test.score <= ERROR_BOUND